In [ ]:
# ============================================================
# STEP 2 — IMAGE FEATURE EXTRACTION (EfficientNetB0 + ViT)
# For CT + US folders with train/valid/test structure
# ============================================================

import os
import numpy as np
from tqdm import tqdm
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from transformers import ViTFeatureExtractor, TFViTModel
from transformers import ViTFeatureExtractor, TFViTModel
import transformers
transformers.logging.set_verbosity_error()
# ----------------------------
# Paths to dataset
# ----------------------------
base_path = "/kaggle/input/classification-dataset1"
ct_base = os.path.join(base_path, "CT_Classification.v1i.folder/CT_Classification.v1i.folder")
us_base = os.path.join(base_path, "Ultrasound_classification.v1i.folder/Ultrasound_classification.v1i.folder")

splits = ["train", "valid", "test"]
os.makedirs("features", exist_ok=True)

# ----------------------------
# EfficientNetB0 (local features)
# ----------------------------
IMG_SIZE = (224, 224)
eff_base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224,224,3))
eff_out = tf.keras.layers.GlobalAveragePooling2D()(eff_base.output)
eff_model = Model(inputs=eff_base.input, outputs=eff_out)

def extract_eff_features(folder):
    image_paths, labels = [], []
    for cls in sorted(os.listdir(folder)):
        cls_dir = os.path.join(folder, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(cls_dir, f))
                labels.append(cls.lower())
    feats = []
    for path in tqdm(image_paths, desc=f"EfficientNet extracting from {folder}"):
        try:
            img = tf.io.read_file(path)
            img = tf.image.decode_image(img, channels=3)
            img = tf.image.resize(img, IMG_SIZE)
            img = tf.cast(img, tf.float32) / 255.0
            feat = eff_model(tf.expand_dims(img, 0), training=False)
            feats.append(feat.numpy().squeeze())
        except Exception as e:
            print("❌ Error reading:", path, "|", e)
    return np.array(feats), np.array(labels)

# ----------------------------
# ViT (global features)
# ----------------------------
vit_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
vit_model = TFViTModel.from_pretrained("google/vit-base-patch16-224")

def extract_vit_features(folder):
    image_paths, labels = [], []
    for cls in sorted(os.listdir(folder)):
        cls_dir = os.path.join(folder, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(cls_dir, f))
                labels.append(cls.lower())
    feats = []
    for path in tqdm(image_paths, desc=f"ViT extracting from {folder}"):
        try:
            img = Image.open(path).convert("RGB").resize((224,224))
            inputs = vit_extractor(images=img, return_tensors="tf")
            outputs = vit_model(**inputs)
            cls_vec = outputs.last_hidden_state[:,0,:].numpy().squeeze()
            feats.append(cls_vec)
        except Exception as e:
            print("❌ Error reading:", path, "|", e)
    return np.array(feats), np.array(labels)

# ----------------------------
# Extract for CT and US (train/valid/test)
# ----------------------------
for split in splits:
    # CT
    ct_split = os.path.join(ct_base, split)
    if os.path.exists(ct_split):
        ct_eff, ct_lbl = extract_eff_features(ct_split)
        ct_vit, _ = extract_vit_features(ct_split)
        np.save(f"features/ct_eff_{split}.npy", ct_eff)
        np.save(f"features/ct_vit_{split}.npy", ct_vit)
        np.save(f"features/ct_labels_{split}.npy", ct_lbl)
        print(f"✅ Saved CT {split}: Eff={ct_eff.shape}, ViT={ct_vit.shape}")

    # Ultrasound
    us_split = os.path.join(us_base, split)
    if os.path.exists(us_split):
        us_eff, us_lbl = extract_eff_features(us_split)
        us_vit, _ = extract_vit_features(us_split)
        np.save(f"features/us_eff_{split}.npy", us_eff)
        np.save(f"features/us_vit_{split}.npy", us_vit)
        np.save(f"features/us_labels_{split}.npy", us_lbl)
        print(f"✅ Saved US {split}: Eff={us_eff.shape}, ViT={us_vit.shape}")

print("\n🎯 All CT + US image features extracted and saved successfully!")

In [ ]:
# ============================================================
# STEP 1 — TABULAR FEATURE EXTRACTION  (CT + US combined)
# ============================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
import os

# =========================
# 1️⃣ Load both CSV files
# =========================
us_csv = "/kaggle/input/csv-classification/Patient History_CSV_US_Combine_Classify.csv"
ct_csv = "/kaggle/input/csv-classification/patient_history_CSV_CT_Combine_Classify.csv"

us_df = pd.read_csv(us_csv)
ct_df = pd.read_csv(ct_csv)

# =========================
# 2️⃣ Standardize column names
# =========================
us_df.columns = us_df.columns.str.strip().str.lower()
ct_df.columns = ct_df.columns.str.strip().str.lower()

# Unify naming differences
us_df = us_df.rename(columns={'type of water': 'type_of_water', 'region': 'region'})
ct_df = ct_df.rename(columns={'type of water': 'type_of_water', 'region': 'region'})

# Add modality indicator
us_df['modality'] = 'us'
ct_df['modality'] = 'ct'

# =========================
# 3️⃣ Keep relevant columns
# =========================
cols = [
    'age', 'gender', 'profession', 'weight',
    'water', 'type_of_water', 'region',
    'class', 'modality'
]
us_df = us_df[cols]
ct_df = ct_df[cols]

# Merge both modalities
tab_df = pd.concat([us_df, ct_df], ignore_index=True)

# =========================
# 4️⃣ Clean missing / nulls
# =========================
tab_df = tab_df.fillna('unknown')

# Normalize label capitalization
tab_df['class'] = tab_df['class'].astype(str).str.strip().str.lower()

# =========================
# 5️⃣ Encode features
# =========================
num_cols = ['age', 'weight']
cat_cols = ['gender', 'profession', 'water', 'type_of_water', 'region', 'modality']

# Scale numeric columns
scaler = StandardScaler()
tab_df[num_cols] = scaler.fit_transform(tab_df[num_cols])

# One-hot encode categorical columns
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_features = enc.fit_transform(tab_df[cat_cols])

# Combine numeric + categorical features
X_tab = np.hstack([tab_df[num_cols].values, cat_features])

# Encode labels
le = LabelEncoder()
y_tab = le.fit_transform(tab_df['class'])

# =========================
# 6️⃣ Save processed arrays
# =========================
os.makedirs("features", exist_ok=True)
np.save("features/tabular_features.npy", X_tab)
np.save("features/labels.npy", y_tab)

# =========================
# 7️⃣ Display summary
# =========================
print("✅ Tabular features saved successfully!")
print("   features/tabular_features.npy  →", X_tab.shape)
print("   features/labels.npy            →", y_tab.shape)
print("   Label classes:", le.classes_)
print("\nClass distribution:\n", tab_df['class'].value_counts())

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

# base features folder (Kaggle default)
feat_dir = "/kaggle/working/features"

In [ ]:
def load_features(modality, split):
    eff = np.load(f"{feat_dir}/{modality}eff{split}.npy")
    vit = np.load(f"{feat_dir}/{modality}vit{split}.npy")
    labels = np.load(f"{feat_dir}/{modality}labels{split}.npy", allow_pickle=True)
    return eff, vit, labels

In [ ]:
import numpy as np
import os

def load_features(modality, split):
    eff = np.load(f"features/{modality}_eff_{split}.npy", allow_pickle=True)
    vit = np.load(f"features/{modality}_vit_{split}.npy", allow_pickle=True)
    lbl = np.load(f"features/{modality}_labels_{split}.npy", allow_pickle=True)
    return eff, vit, lbl

In [ ]:
splits = ["train", "valid", "test"]
all_eff, all_vit, all_lbl = {}, {}, {}

for split in splits:
    eff_ct, vit_ct, lbl_ct = load_features("ct", split)
    eff_us, vit_us, lbl_us = load_features("us", split)
    all_eff[split] = np.vstack([eff_ct, eff_us])
    all_vit[split] = np.vstack([vit_ct, vit_us])
    all_lbl[split] = np.concatenate([lbl_ct, lbl_us])

for s in splits:
    print(f"{s:>5} | Eff {all_eff[s].shape} | ViT {all_vit[s].shape} | n={len(all_lbl[s])}")

In [ ]:
X_tab = np.load(f"{feat_dir}/tabular_features.npy")
y_tab = np.load(f"{feat_dir}/labels.npy")
print("Tabular:", X_tab.shape, "Labels:", y_tab.shape)

In [ ]:
def sample_tabular(n):
    idx = np.random.randint(0, X_tab.shape[0], n)
    return X_tab[idx]

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform([l.lower() for l in all_lbl["train"]])
y_val_enc   = le.transform([l.lower() for l in all_lbl["valid"]])
y_test_enc  = le.transform([l.lower() for l in all_lbl["test"]])

X_train = np.concatenate([
    all_eff["train"],
    all_vit["train"],
    sample_tabular(len(y_train_enc))
], axis=1)

X_val = np.concatenate([
    all_eff["valid"],
    all_vit["valid"],
    sample_tabular(len(y_val_enc))
], axis=1)

X_test = np.concatenate([
    all_eff["test"],
    all_vit["test"],
    sample_tabular(len(y_test_enc))
], axis=1)

print("Final shapes → Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Classes:", le.classes_)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import joblib   # for saving scaler

# ----------------------------
# 1️⃣ Feature Scaling
# ----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Save scaler (IMPORTANT)
joblib.dump(scaler, "scaler.pkl")

# ----------------------------
# 2️⃣ Build Model
# ----------------------------
input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train_enc))

model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(num_classes, activation='softmax')
])

# ----------------------------
# 3️⃣ Compile Model
# ----------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ----------------------------
# 4️⃣ Callbacks (Save Best Model in .h5)
# ----------------------------
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "kidney_best_model.h5",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=3,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=7,
        restore_best_weights=True
    )
]

# ----------------------------
# 5️⃣ Train Model
# ----------------------------
history = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ----------------------------
# 6️⃣ Evaluate Model
# ----------------------------
train_loss, train_acc = model.evaluate(X_train, y_train_enc, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val_enc, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test_enc, verbose=0)

print(f"\nTrain Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Detailed report
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test_enc, y_pred_classes))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test_enc, y_pred_classes))

# ----------------------------
# 7️⃣ Save Final Model (.h5)
# ----------------------------
model.save("kidney_final_model.h5")
print("\nModel saved successfully in .h5 format ✅")

In [ ]:
import tensorflow as tf
import joblib

# Load model
model = tf.keras.models.load_model("kidney_final_model.h5")

# Load scaler
scaler = joblib.load("scaler.pkl")

print("Model and scaler loaded successfully ✅")

In [ ]:
import matplotlib.pyplot as plt

# Loss
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Validation Accuracy vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np


plt.figure(figsize=(6,5))

sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Cyst','Normal','Stone','Tumor'],
            yticklabels=['Cyst','Normal','Stone','Tumor'])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# --- Your input image ---
img_path = "/kaggle/input/classification-dataset1/CT_Classification.v1i.folder/CT_Classification.v1i.folder/test/kidney_normal/axial-74-_jpeg.rf.78d32968a807bdfe9ef694f491193327.jpg"

# Read image (BGR → RGB)
image = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# --- Extract features ---
eff_feat, vit_feat = extract_image_features(img_path)

# Dummy tabular vector
tab_dim = 680
tab_feat = np.zeros(tab_dim)

# Concatenate all features
x_input = np.concatenate([eff_feat, vit_feat, tab_feat])[np.newaxis, :]

# --- Prediction ---
pred = model.predict(x_input)
class_idx = np.argmax(pred)
prob = np.max(pred)
classes = ['cyst', 'normal', 'stone', 'tumor']
predicted_label = f"{classes[class_idx].upper()} ({prob:.2f})"

print("Predicted:", predicted_label)

# --- Overlay prediction on image ---
output_img = image_rgb.copy()
cv2.putText(
    output_img,
    predicted_label,
    (20, 40),                # position
    cv2.FONT_HERSHEY_SIMPLEX,
    1.0,                     # font scale
    (255, 0, 0),             # color (Blue-Red-Green)
    2,                       # thickness
    cv2.LINE_AA
)

# --- Display image ---
plt.figure(figsize=(7,7))
plt.imshow(output_img)
plt.axis("off")
plt.title("Predicted Output")
plt.show()

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

img_path = r"/kaggle/input/classification-dataset1/CT_Classification.v1i.folder/CT_Classification.v1i.folder/test/kidney_normal/axial-70-_jpeg.rf.61ca9d03c46033dd5ab1f860dabaddae.jpg"

# Load Image
image = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Get center coordinates
h, w = image_rgb.shape[:2]
center = (w // 2, h // 2)
radius = min(h, w) // 4  # adjustable

# Draw circle
output_img = image_rgb.copy()
cv2.circle(output_img, center, radius, (255, 0, 0), 3)

# Add predicted label
cv2.putText(output_img, predicted_label, (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
            (255, 0, 0), 2, cv2.LINE_AA)

plt.figure(figsize=(7,7))
plt.imshow(output_img)
plt.axis("off")
plt.show()


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread("/kaggle/input/classification-dataset1/CT_Classification.v1i.folder/CT_Classification.v1i.folder/test/kidney_normal/axial-70-_jpeg.rf.61ca9d03c46033dd5ab1f860dabaddae.jpg", 0)  # grayscale
image_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# Threshold to find bright kidney region
_, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Find contours
contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Largest contour (kidney area)
cnt = max(contours, key=cv2.contourArea)

# Minimum enclosing circle
(x, y), radius = cv2.minEnclosingCircle(cnt)
center = (int(x), int(y))
radius = int(radius)

# Draw circle
output = image_rgb.copy()
cv2.circle(output, center, radius, (255, 0, 0), 3)

# Add predicted label
cv2.putText(output, predicted_label, (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
            (255, 0, 0), 2)

plt.figure(figsize=(7,7))
plt.imshow(output)
plt.axis("off")
plt.show()
